## Install

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
import torchvision.transforms as transforms
from torchvision.utils import save_image
from tqdm import tqdm
import numpy as np
import os
import shutil
from pytorch_fid import fid_score
import medmnist
from medmnist import INFO
import itertools
import json
import time

# Configurações globais
BASE_CONFIG = {
    'data_flag': 'bloodmnist',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'batch_size': 64,
    'num_epochs': 200,
    'image_size': 32,
    'output_root': "./experiments",
    'fid': {
        'num_samples': 10000,
        'batch_size': 50,
        'dims': 2048,
        'seeds': [42, 123, 456, 789, 999],
    }
}

# Hiperparâmetros para grid search
GRID_PARAMS = {
    'latent_dim': [64],
    'lr': [0.0001],
    'beta1': [(0.5, 0.999)]
}

if BASE_CONFIG['device'] == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()
    print(f"GPU disponível: {torch.cuda.get_device_name(0)}")

## Data

In [ ]:
def prepare_data():
    transform = transforms.Compose([
        transforms.Resize(BASE_CONFIG['image_size']),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    DataClass = getattr(medmnist, INFO[BASE_CONFIG['data_flag']]['python_class'])
    train = DataClass(split='train', transform=transform, download=True)
    val = DataClass(split='val', transform=transform, download=True)
    test = DataClass(split='test', transform=transform, download=True)

    return ConcatDataset([train, val, test])


#

## Generator


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 3, 4, 2, 1, bias=False)
        )

    def forward(self, x):
        x = self.main(x)
        return torch.clamp(torch.tanh(x), -1, 1)

## Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x).view(-1)

## Treino

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class GANTrainer:
    def __init__(self, config, dataloader):
        self.config = config
        self.device = config['device']
        self.dataloader = dataloader

        self.generator = Generator(config['latent_dim']).to(self.device)
        self.discriminator = Discriminator().to(self.device)
        self.generator.apply(weights_init)
        self.discriminator.apply(weights_init)

        self.optimizerG = optim.AdamW(
            self.generator.parameters(), 
            lr=config['lr'], 
            betas=config['beta1']
        )
        self.optimizerD = optim.AdamW(
            self.discriminator.parameters(), 
            lr=config['lr'], 
            betas=config['beta1']
        )
        self.criterion = nn.BCELoss()
        
    def train(self):
        for epoch in range(self.config['num_epochs']):
            self._train_epoch(epoch)

            if (epoch+1) % 10 == 0 or (epoch+1) == self.config['num_epochs']:
                self._save_samples(epoch+1)
                self._save_checkpoint(epoch+1)

    def _train_epoch(self, epoch):
        self.generator.train()
        self.discriminator.train()

        progress_bar = tqdm(self.dataloader, desc=f'Epoch {epoch+1}/{self.config["num_epochs"]}')
        for real_imgs, _ in progress_bar:
            real_imgs = real_imgs.to(self.device)
            batch_size = real_imgs.size(0)

            real_labels = torch.full((batch_size,), 0.9, device=self.device)
            fake_labels = torch.zeros(batch_size, device=self.device)

            # Treina Discriminador
            self.optimizerD.zero_grad()
            noise = torch.randn(batch_size, self.config['latent_dim'], 1, 1, device=self.device)
            fake_imgs = self.generator(noise)
            output_real = self.discriminator(real_imgs)
            loss_real = self.criterion(output_real, real_labels)
            output_fake = self.discriminator(fake_imgs.detach())
            loss_fake = self.criterion(output_fake, fake_labels)
            loss_D = loss_real + loss_fake
            loss_D.backward()
            self.optimizerD.step()

            # Treina Gerador (2x)
            for _ in range(2):
                self.optimizerG.zero_grad()
                noise = torch.randn(batch_size, self.config['latent_dim'], 1, 1, device=self.device)
                fake_imgs = self.generator(noise)
                output = self.discriminator(fake_imgs)
                loss_G = self.criterion(output, real_labels)
                loss_G.backward()
                self.optimizerG.step()

            progress_bar.set_postfix({
                'Loss D': f'{loss_D.item():.4f}', 
                'Loss G': f'{loss_G.item():.4f}'
            })

    def _save_samples(self, epoch, num_samples=64):
        self.generator.eval()
        with torch.no_grad():
            noise = torch.randn(num_samples, self.config['latent_dim'], 1, 1, device=self.device)
            samples = self.generator(noise)
            samples = (samples + 1) / 2
            
            sample_dir = os.path.join(self.config['exp_dir'], 'samples')
            os.makedirs(sample_dir, exist_ok=True)
            save_image(samples, os.path.join(sample_dir, f'epoch_{epoch}.png'), nrow=8)

    def _save_checkpoint(self, epoch):
        ckpt_dir = os.path.join(self.config['exp_dir'], 'checkpoints')
        os.makedirs(ckpt_dir, exist_ok=True)
        
        torch.save({
            'epoch': epoch,
            'generator_state_dict': self.generator.state_dict(),
            'discriminator_state_dict': self.discriminator.state_dict(),
            'optimizerG_state_dict': self.optimizerG.state_dict(),
            'optimizerD_state_dict': self.optimizerD.state_dict(),
            'config': self.config
        }, os.path.join(ckpt_dir, f'gan_epoch_{epoch}.pth'))

## Gerar 10k Imagens

In [ ]:
def generate_images(generator, device, output_dir, num_images=10000):
    """Gera imagens sintéticas e salva em disco"""
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    generator.eval()
    with torch.no_grad():
        for i in tqdm(range(0, num_images, CONFIG['batch_size']), desc="Generating images"):
            batch_size = min(CONFIG['batch_size'], num_images - i)
            noise = torch.randn(batch_size, CONFIG['latent_dim'], 1, 1, device=device)
            fake_images = generator(noise)
            fake_images = (fake_images + 1) / 2  # Scale from [-1,1] to [0,1]
            
            for j in range(batch_size):
                save_image(fake_images[j], os.path.join(output_dir, f"{i+j}.png"))

## Salvar 10k Imagens

In [ ]:
def save_real_images(dataset, output_dir, num_images=10000):
    """Salva imagens reais para cálculo do FID"""
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    count = 0
    for image, _ in tqdm(dataset, desc="Saving real images", total=num_images):
        if count >= num_images:
            break
        save_image(image, os.path.join(output_dir, f"{count}.png"))
        count += 1

## FID

In [ ]:
def evaluate_fid(generator, dataset, config):
    """Calcula FID usando apenas a seed de treinamento"""
    try:
        # Preparar diretórios
        real_dir = os.path.join(config['exp_dir'], 'fid_real')
        fake_dir = os.path.join(config['exp_dir'], 'fid_fake')
        
        # Limpar diretórios existentes
        for d in [real_dir, fake_dir]:
            if os.path.exists(d):
                shutil.rmtree(d)
            os.makedirs(d, exist_ok=True)
        
        # Salvar imagens reais
        print("Salvando imagens reais...")
        count = 0
        for img, _ in tqdm(dataset, total=config['fid']['num_samples']):
            if count >= config['fid']['num_samples']:
                break
            save_image(img, os.path.join(real_dir, f'{count}.png'))
            count += 1
        
        # Usar a mesma seed do treinamento para geração
        seed = config['seed']
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        print(f"Gerando imagens sintéticas com seed {seed}...")
        generator.eval()
        with torch.no_grad():
            for i in tqdm(range(0, config['fid']['num_samples'], config['batch_size'])):
                batch_size = min(config['batch_size'], config['fid']['num_samples'] - i)
                noise = torch.randn(batch_size, config['latent_dim'], 1, 1, device=config['device'])
                fake_imgs = generator(noise)
                fake_imgs = (fake_imgs + 1) / 2
                
                for j in range(batch_size):
                    save_image(fake_imgs[j], os.path.join(fake_dir, f'{i+j}.png'))
        
        print("Calculando FID...")
        fid = fid_score.calculate_fid_given_paths(
            [real_dir, fake_dir],
            batch_size=config['fid']['batch_size'],
            device=config['device'],
            dims=config['fid']['dims'],
            num_workers=0  # Evitar problemas com multiprocessamento
        )
        
        print(f"FID com seed {seed}: {fid:.2f}")
        return [fid], fid, 0.0  # Retornar como lista para consistência
        
    except Exception as e:
        print(f"Erro no cálculo do FID: {str(e)}")
        return [float('nan')], float('nan'), float('nan')
    finally:
        # Limpeza
        torch.cuda.empty_cache()

## MAIN

In [ ]:
def run_experiment(params, dataset):
    """Executa um experimento completo com configuração específica"""
    # Criar configuração combinada
    config = {**BASE_CONFIG, **params}
    
    # Diretório raiz do experimento
    base_exp_dir = os.path.join(config['output_root'], f"latent{config['latent_dim']}_lr{config['lr']}_b1{config['beta1'][0]}")
    os.makedirs(base_exp_dir, exist_ok=True)
    
    all_results = []
    all_fids = []
    # Loop por cada seed de TREINAMENTO
    for seed in config['fid']['seeds']:
        # Configuração específica para esta seed
        seed_config = config.copy()
        seed_config['seed'] = seed
        exp_name = f"seed_{seed}"
        seed_config['exp_dir'] = os.path.join(base_exp_dir, exp_name)
        os.makedirs(seed_config['exp_dir'], exist_ok=True)
        
        print(f"\n{'-'*50}")
        print(f"INICIANDO EXPERIMENTO COMPLETO COM SEED: {seed}")
        print(f"{'-'*50}\n")
        
        # Salvar configuração
        with open(os.path.join(seed_config['exp_dir'], 'config.json'), 'w') as f:
            json.dump(seed_config, f, indent=2)
        
        # Fixar seed para reprodutibilidade TOTAL
        torch.manual_seed(seed)
        np.random.seed(seed)
        if seed_config['device'] == 'cuda':
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        
        # Preparar dataloader COM seed fixa
        dataloader = DataLoader(
            dataset, 
            batch_size=seed_config['batch_size'], 
            shuffle=True,
            num_workers=4,
            pin_memory=True,
            generator=torch.Generator().manual_seed(seed)  # Seed para shuffle
        )
        
        # Treinar modelo DO ZERO
        print(f"Treinando modelo com seed {seed}...")
        start_time = time.time()
        trainer = GANTrainer(seed_config, dataloader)
        trainer.train()
        training_time = time.time() - start_time
        
        # Avaliar FID (usando a mesma seed para geração)
        print(f"Avaliando FID com seed {seed}...")
        fid_results, avg_fid, std_fid = evaluate_fid(trainer.generator, dataset, seed_config)
        
        # Salvar resultados
        results = {
            'exp_name': exp_name,
            'seed': seed,
            'fid_results': fid_results,
            'avg_fid': avg_fid,
            'std_fid': std_fid,
            'training_time': training_time,
            'config': seed_config
        }
        
        with open(os.path.join(seed_config['exp_dir'], 'results.json'), 'w') as f:
            json.dump(results, f, indent=2)
        
        print(f"\n{'='*50}")
        print(f"EXPERIMENTO COMPLETO COM SEED {seed} CONCLUÍDO!")
        print(f"FID médio: {avg_fid:.2f} ± {std_fid:.2f}")
        print(f"Tempo total: {training_time/60:.2f} minutos")
        print(f"{'='*50}\n")
        
        all_fids.append(avg_fid)
        all_results.append(results)
        
        # Liberar memória
        del trainer
        torch.cuda.empty_cache()
        
        if all_fids:
            final_avg_fid = np.mean(all_fids)
            final_std_fid = np.std(all_fids)
            
            # Salva um arquivo de resumo FINAL
            summary = {
                'params': params,
                'fids_all_seeds': all_fids,
                'final_avg_fid': final_avg_fid,
                'final_std_fid': final_std_fid,
                'all_results': all_results
            }
            
            summary_path = os.path.join(base_exp_dir, "summary.json")
            with open(summary_path, 'w') as f:
                json.dump(summary, f, indent=2)
            
            print(f"\n{'#'*50}")
            print(f"RESUMO FINAL PARA {base_exp_dir}:")
            print(f"FIDs individuais: {all_fids}")
            print(f"FID MÉDIO: {final_avg_fid:.2f} ± {final_std_fid:.2f}")
            print(f"{'#'*50}\n")
    
    return all_results

def grid_search():
    """Executa busca em grade nos hiperparâmetros"""
    # Preparar dados uma única vez
    dataset = prepare_data()
    
    # Gerar combinações de parâmetros
    param_grid = [
        dict(zip(GRID_PARAMS.keys(), values)) 
        for values in itertools.product(*GRID_PARAMS.values())
    ]
    
    print(f"Total de experimentos: {len(param_grid)}")
    print(f"Parâmetros testados: {GRID_PARAMS}")
    
    all_results = []
    best_fid = float('inf')
    best_config = None
    
    for i, params in enumerate(param_grid):
        print(f"\n{'='*80}")
        print(f"EXPERIMENTO {i+1}/{len(param_grid)}")
        print(f"Parâmetros: {params}")
        
        try:
            results = run_experiment(params, dataset)
            all_results.append(results)
            
            # Atualizar melhor resultado
            if results['avg_fid'] < best_fid:
                best_fid = results['avg_fid']
                best_config = results['config']
                print(f"Novo melhor FID: {best_fid:.2f}")
                
        except Exception as e:
            print(f"Erro no experimento {params}: {str(e)}")
            continue
    
    # Salvar resultados consolidados
    summary = {
        'best_fid': best_fid,
        'best_config': best_config,
        'all_results': all_results
    }
    
    summary_path = os.path.join(BASE_CONFIG['output_root'], 'grid_search_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print("\nBusca em grade concluída!")
    print(f"Melhor FID: {best_fid:.2f}")
    print(f"Melhor configuração: {best_config}")
    print(f"Resumo salvo em: {summary_path}")
    
    return summary

if __name__ == "__main__":
    # Executar grid search
    grid_search()